<a href="https://colab.research.google.com/github/liz-lewis-manchester/CNM_2025_group_09/blob/Test-2%2C-2.4-and-2.5/Coursework9.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Computing & Numerical Methods Coursework Group 9
By: Ching Yau Chan, Hassan Alhamdani, Jiongjie Chen, Lucas So and Oyinmiebi Youdeowei

# Test 1

# Test 1.3

# Test 1.4

# Test 1.5

# Test 2

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation

# These are the inputs that act as parameters that allow the user to specify model domain, resolution, and boundary conditions at the edge of the domain
end_time = float(input("How long should the model last for, in seconds  "))
time_interval = float(input("The time interval, in seconds   "))
length = float(input("Length of model, in metres   "))
length_interval = float(input("The length interval, in metres   "))
speed = float(input("Speed of river flow, in m/s   "))
initial_concentration = str(input("Name of the .csv file   "))


def concentration_from_csv(end_time, time_interval, length, length_interval, speed, initial_concentration):

    # Creates an empty array to hold 'distance' values
    distance = np.array([])

    # This calculates how many steps in distance that can fit within the interval and loops that many times
    for i in range(int(length / length_interval + 1)):
        # Adds the calculated values for distance step into the 'distance' array
        distance = np.append(distance, length_interval * i)

    # Reads the csv inputted previously, using 'latin1' encoding for more compatibility and puts it into a dataframe
    df = pd.read_csv(initial_concentration, encoding='latin1')

    # Inputs the values for distance and concentration from the dataframe into separate arrays
    x_data = df["Distance (m)"].values
    C_data = df["Concentration (µg/m_ )"].values

    # Interpolates the initial concentration data onto the defined distance grid and values outside the range are set to 0
    initial_conditions = np.interp(distance, x_data, C_data, left=0.0, right=0.0)

    fig, ax = plt.subplots()

    # Calculates the coefficients A and B for the concentration model equation
    A_value = 1 / time_interval + speed / length_interval
    B_value = speed / length_interval

    # Creates arrays for A and B
    A_array = np.array([])
    B_array = np.array([])

    for i in range(int(length / length_interval + 1)):
        A_array = np.append(A_array, A_value)
        B_array = np.append(B_array, B_value)

    # Calculates the total number of frames (time steps) for the animation
    num_frames = (int(end_time / time_interval)+1)

    # Creates a copy of the initial conditions to be updated in each time step
    current_initial_conditions = initial_conditions.copy()

    # List to store concentration distributions at each time step
    all_concentrations = []

    # Loop to calculate concentration at each time step
    for k in range(num_frames):
        if k == 0:
            # Appends initial conditions for the first frame
            all_concentrations.append(initial_conditions.copy())
        else:
            # Initialize an array for the concentration at the current time step
            concentration_present = np.zeros(int(length / length_interval + 1))
            concentration_present[0] = df.iloc[0, 1]

            # Calculate concentration for subsequent points using the numerical model
            for i in range(1, int(length / length_interval + 1)):
                concentration_present[i] = (current_initial_conditions[i] / time_interval + B_array[i] * concentration_present[i - 1]) / A_array[i]

            # Update current_initial_conditions for the next time step
            current_initial_conditions = concentration_present.copy()
            # Store the concentration profile for the current time step
            all_concentrations.append(concentration_present.copy())

    # Sets animation to display in HTML
    plt.rcParams["animation.html"] = "jshtml"
    plt.rcParams['figure.dpi'] = 150
    plt.ioff()

    def animate(f):
        plt.cla() # Clear the current axes

        # Get the concentration profile for the current frame
        concentration = all_concentrations[f]

        plt.plot(distance, concentration)

        # Determines the maximum initial concentration for setting plot y-axis limits
        max_C0 = np.max(initial_conditions)

        # Set plot limits, labels, title, and grid
        plt.xlim(0, length)
        plt.ylim(0, max_C0 * 1.1 if max_C0 > 0 else 1.0) # Y-axis range is maximum concentration multiplied by 1.1 to keep whole curve in frame
        plt.xlabel("Distance (m)")
        plt.ylabel("Concentration (μg/m³)")
        plt.title(f"Test 2: Concentration at Time = {f * time_interval:.1f} s")
        plt.grid(True)

    # Create and return the animation object
    anim = matplotlib.animation.FuncAnimation(fig, animate, frames=num_frames)
    return anim

# Call the function to generate the animation
anim = concentration_from_csv(end_time, time_interval, length, length_interval, speed, initial_concentration)
anim # Display the animation

# Test 2.3

# Test 2.4

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation

# These are the inputs that act as parameters that allow the user to specify model domain, resolution, and boundary conditions at the edge of the domain
end_time = float(input("How long should the model last for, in seconds  "))
time_interval = float(input("The time interval, in seconds   "))
length = float(input("Length of model, in metres   "))
length_interval = float(input("The length interval, in metres   "))
speed = float(input("Speed of river flow, in m/s   "))
initial_concentration = str(input("Name of the .csv file   "))
# Input for decay constant has been added
decay_constant = float(input("Decay constant for exponential decay (e.g., 0.01), enter 0 for no decay   "))

def concentration_from_csv(end_time, time_interval, length, length_interval, speed, initial_concentration, decay_constant):
    distance = np.array([])# Creates an empty array to hold 'distance' values
    for i in range(int(length / length_interval + 1)):
        distance = np.append(distance, length_interval * i)

    # Read the initial condition data in the CSV file, using "latin1" to make sure the correct format of the file
    df = pd.read_csv(initial_concentration, encoding='latin1')

    # Get the datas from the CSV file
    x_data = df["Distance (m)"].values
    C_data = df["Concentration (µg/m_ )"].values

    # Interpolate the concentration onto the grid, Set the concentration over measurement range to 0
    initial_conditions = np.interp(distance, x_data, C_data, left=0.0, right=0.0)

    # Extract the initial pollutant concentration at position x = 0 from the CSV file and save it as C0_initial
    C0_initial = df.iloc[0, 1]


    # Creates a figure and an axes object for plotting
    fig, ax = plt.subplots()

    # Calculate the coefficients A and B to make the later calculation easier
    A_value = 1 / time_interval + speed / length_interval
    B_value = speed / length_interval

    # Create empty arrays for coefficients A and B
    A_array = np.array([])
    B_array = np.array([])

    # Input the values of A and B into the assigned arrays
    for i in range(int(length / length_interval + 1)):
        A_array = np.append(A_array, A_value)
        B_array = np.append(B_array, B_value)

    # Calculate the total number of frames
    num_frames = (int(end_time / time_interval)+1)

    # Record the current concentration distribution
    current_initial_conditions = initial_conditions.copy()

    all_concentrations = []

    # Calculate the concentration for each time and store it in the all_concentrations array
    for k in range(num_frames):
        if k == 0:
            all_concentrations.append(initial_conditions.copy())
        else:
            concentration_present = np.zeros(int(length / length_interval + 1))
            current_time = k * time_interval
            # Apply exponential decay to the boundary concentration at x=0
            concentration_present[0] = C0_initial * np.exp(-decay_constant * current_time) # addition
            # concentration_present[0] = df.iloc[0, 1] -> deleted010

            for i in range(1, int(length / length_interval + 1)):
                concentration_present[i] = (current_initial_conditions[i] / time_interval + B_array[i] * concentration_present[i - 1]) / A_array[i]

            current_initial_conditions = concentration_present.copy()
            all_concentrations.append(concentration_present.copy())

    # Set the animation parameters
    plt.rcParams["animation.html"] = "jshtml"
    plt.rcParams['figure.dpi'] = 150
    plt.ioff()# Turn off interactions to avoid conflicts between animations and interactions

    def animate(f):
        plt.cla()

        concentration = all_concentrations[f]# Find the concentration distribution corresponding to Frame f

        plt.plot(distance, concentration)# Draw the concentration-distance curve

        # Determines the maximum initial concentration for setting plot y-axis limits
        max_C0 = np.max(initial_conditions)

        # Set plot limits, labels, title, and grid
        plt.xlim(0, length)
        max_C0 = np.max(initial_conditions)
        plt.ylim(0, max_C0 * 1.1 if max_C0 > 0 else 1.0) # Y-axis range is maximum concentration multiplied by 1.1 to keep whole curve in frame
        plt.xlabel("Distance (m)")
        plt.ylabel("Concentration (μg/m³)")
        plt.title(f"Test 2.4: Concentration at Time = {f * time_interval:.1f} s (decay = {decay_constant})") # addition: Title now includes decay constant
        plt.grid(True)
        plt.close()

    anim = matplotlib.animation.FuncAnimation(fig, animate, frames=num_frames)# Create animation
    return anim

# Call the function to generate and display the animation with the decay constant
anim = concentration_from_csv(end_time, time_interval, length, length_interval, speed, initial_concentration, decay_constant)
anim # Play and display the animation

# Test 2.5

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.animation

# These are the inputs that act as parameters that allow the user to specify model domain, resolution, and boundary conditions at the edge of the domain
end_time = float(input("How long should the model last for, in seconds  "))
time_interval = float(input("The time interval, in seconds   "))
length = float(input("Length of model, in metres   "))
length_interval = float(input("The length interval, in metres   "))
speed = float(input("Speed of river flow, in m/s   "))
# Input for random pertubation has been added
random_perturbation = float(input("The minimum and maximum of random perturbation of the stream, in percentages   "))
initial_concentration = str(input("Name of the .csv file   "))


def concentration_from_csv(end_time, time_interval, length, length_interval, speed, initial_concentration):

    # Creates an empty array to hold 'distance' values
    distance = np.array([])

    # This calculates how many steps in distance that can fit within the interval and loops that many times
    for i in range(int(length / length_interval + 1)):
        # Adds the calculated values for distance step into the 'distance' array
        distance = np.append(distance, length_interval * i)

    # Reads the csv inputted previously, using 'latin1' encoding for more compatibility and puts it into a dataframe
    df = pd.read_csv(initial_concentration, encoding='latin1')

    # Inputs the values for distance and concentration from the dataframe into separate arrays
    x_data = df["Distance (m)"].values
    C_data = df["Concentration (µg/m_ )"].values

    # Interpolates the initial concentration data onto the defined distance grid and values outside the range are set to 0
    initial_conditions = np.interp(distance, x_data, C_data, left=0.0, right=0.0)

    # Creates a figure and an axes object for plotting
    fig, ax = plt.subplots()


    # Calculates the coefficients A and B for the concentration model equation
    A_value = 1 / time_interval + speed / length_interval
    B_value = speed / length_interval

    # Creates arrays for A and B
    A_array = np.array([])
    B_array = np.array([])

    for i in range(int(length / length_interval + 1)):
        A_array = np.append(A_array, A_value)
        B_array = np.append(B_array, B_value)

    # Calculates the total number of frames (time steps) for the animation
    num_frames = (int(end_time / time_interval)+1)

    # Creates a copy of the initial conditions to be updated in each time step
    current_initial_conditions = initial_conditions.copy()

    # List to store concentration distributions at each time step
    all_concentrations = []

    # Loop to calculate concentration at each time step
    for k in range(num_frames):
        if k == 0:
            all_concentrations.append(initial_conditions.copy())
        else:
            concentration_present = np.zeros(int(length / length_interval + 1))
            concentration_present[0] = df.iloc[0, 1]

            # Generates a random value for each segment of the length interval.
            random_variable = np.random.random((int(length / length_interval+1)))
            # Calculate the speed with random perturbation applied
            random_speed = (1 - random_perturbation/100 + random_perturbation/50 * random_variable) * speed

            # Random speed affects A and B values in the concentration calculation
            A_value = 1/time_interval + random_speed / length_interval
            B_value = random_speed / length_interval

            for i in range(1, int(length / length_interval + 1)):
                # Calculate concentration for the current point using the perturbed speed's A and B values
                concentration_present[i] = (current_initial_conditions[i] / time_interval + B_value[i] * concentration_present[i - 1]) / A_value[i]

            current_initial_conditions = concentration_present.copy()
            all_concentrations.append(concentration_present.copy())

    # Sets animation to display in HTML
    plt.rcParams["animation.html"] = "jshtml"
    plt.rcParams['figure.dpi'] = 150
    plt.ioff()

    def animate(f):
        plt.cla() # Clear the current axes

        # Get the concentration for the current frame
        concentration = all_concentrations[f]

        plt.plot(distance, concentration)

        # Determines the maximum initial concentration for setting plot y-axis limits
        max_C0 = np.max(initial_conditions)

        # Set plot limits, labels, title, and grid
        plt.xlim(0, length)
        plt.ylim(0, max_C0 * 1.1 if max_C0 > 0 else 1.0)
        plt.xlabel("Distance (m)")
        plt.ylabel("Concentration (μg/m³)")
        # Title now includes information about velocity perturbation
        plt.title(f"Test 2.5: Concentration at Time = {f * time_interval:.1f} s (Velocity perturbation: {random_perturbation}%)")
        plt.grid(True)

    # Create and return the animation object
    anim = matplotlib.animation.FuncAnimation(fig, animate, frames=num_frames)
    return anim

# Call the function to generate the animation
anim = concentration_from_csv(end_time, time_interval, length, length_interval, speed, initial_concentration)
anim # Display the animation



The random perturbation on speed is most visible when there is a consistent concentration decrease over distance (e.g. t = 100s) instead of stagnation (e.g. t = 270s) in concentration. Interestingly, at the early stages of the graph, the bumpiness of the curve is more visible compared to the late stages of the graph, mostly because the initial concentration does not decrease over distance. Compared to the graph for consistent speed, the lines are bumpy and not smooth due to slight increase or decrease in rate of change of concentration from fluctuating speed.